# Titanic Dataset — GroupBy, Aggregations and Merge

## Objective

The objective of this notebook is to compare different passenger groups using pandas aggregation methods and to introduce the combination of related datasets through merge operations.

In [1]:
# Import of modules and load of the dataset

import pandas as pd
from IPython.display import display

titanic = pd.read_csv("../data/train.csv")

display(titanic.head())
print("Dataset shape:", titanic.shape)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Dataset shape: (891, 12)


## Summary Statistics

Before comparing passenger groups, the following checks provide a descriptive overview of the main numerical and categorical variables.

In [2]:
# 1. Generate statistical description of numeric columns

display(titanic.describe())

# 2. Count male and female passengers

print(titanic["Sex"].value_counts())

# 3. Count passengers for every class

print(titanic["Pclass"].value_counts())

# 4. Count passengers for port of embarkation

print(titanic["Embarked"].value_counts())

# 5. Show the distribution of the 'Survived' column

survival_distribution = pd.DataFrame({"passenger_count": titanic["Survived"].value_counts().sort_index(),
    "percentage": (titanic["Survived"].value_counts(normalize=True).sort_index().mul(100).round(2)
    )
})
display(survival_distribution)

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


Sex
male      577
female    314
Name: count, dtype: int64
Pclass
3    491
1    216
2    184
Name: count, dtype: int64
Embarked
S    644
C    168
Q     77
Name: count, dtype: int64


,passenger_count,percentage
Survived,,
0,549,61.62
1,342,38.38


## Observations

The previous results support the following initial observations:
- Third class passengers were the most present one (491), followed by first class passengers (216) and second class lastly (184)
- Male passengers were the majority (577 vs 314 female passengers)
- The most frequent port of embarkation was Southampton (644 passengers), followed by Cherbourg (168) and Queenstown (77)
- The mean of the `Survived` column tells us that only about 38% of the passengers survived

## Grouped Analysis

The following analyses compare survival outcomes and passenger characteristics across different groups.

In [3]:
# 1. Passenger count, survivor count and survival rate by sex

survival_by_sex = titanic.groupby("Sex").agg(
    passenger_count = ("Sex", "size"),
    survivor_count = ("Survived", "sum"),
    survival_rate = ("Survived", "mean")
)

survival_by_sex = survival_by_sex.round({
    "survival_rate": 3
})

display(survival_by_sex)

,passenger_count,survivor_count,survival_rate
Sex,,,
female,314,233,0.742
male,577,109,0.189


In [4]:
# 2. Compute rate of survival for passenger's class sorting them from higher to lower

titanic.groupby("Pclass")["Survived"].mean().sort_values(ascending = False)

Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

In [5]:
# 3. Group by sex and passenger's class and compute the rate of survival

titanic.groupby(["Sex", "Pclass"])["Survived"].mean()

Sex     Pclass
female  1         0.968085
        2         0.921053
        3         0.500000
male    1         0.368852
        2         0.157407
        3         0.135447
Name: Survived, dtype: float64

In [6]:
# 4. For who survived compute: number of passengers, average age, median age, average fare
# Missing age values are excluded from the calculation. Therefore, the resulting averages describe only passengers with an observed age and may not fully represent the entire group.

survival_summary = titanic.groupby("Survived").agg(
    passenger_count=("Survived", "size"),
    observed_age_count=("Age", "count"),
    average_age=("Age", "mean"),
    median_age=("Age", "median"),
    average_fare=("Fare", "mean"),
)

survival_summary = survival_summary.round({
    "average_age": 2,
    "median_age": 2,
    "average_fare": 2
})

display(survival_summary)

,passenger_count,observed_age_count,average_age,median_age,average_fare
Survived,,,,,
0,549,424,30.63,28.0,22.12
1,342,290,28.34,28.0,48.40


In [7]:
# 5. Compute a single table containing, for every class: passenger_count, survival_rate, average_age, average_fare

class_summary = titanic.groupby("Pclass").agg( 
    passenger_count = ("Pclass", "size"), 
    survival_rate = ("Survived", "mean"), 
    average_age = ("Age", "mean"), 
    average_fare = ("Fare", "mean"))

class_summary = class_summary.round({
    "survival_rate": 3,
    "average_age": 2,
    "average_fare": 2
})

display(class_summary)

,passenger_count,survival_rate,average_age,average_fare
Pclass,,,,
1,216,0.630,38.23,84.15
2,184,0.473,29.88,20.66
3,491,0.242,25.14,13.68


## Observations

### 1. Why is the absolute count of survivors not enough?

Absolute counts may be misleading when comparing groups of different sizes. For example, 20 survivors out of 30 passengers represent a very different outcome from 20 survivors out of 100 passengers. Survival rates allow groups of different sizes to be compared more meaningfully.

### 2. Which variable seems to have a stronger relationship with survival: sex or passenger class?

The observed survival rates suggest that sex has the stronger relationship with survival. The difference between female and male survival rates is particularly large.

Passenger class is also associated with survival, as first-class passengers had a higher survival rate than second- and third-class passengers. However, these results describe associations and do not establish causal relationships.

### 3. Does the higher average survivor rate show that paying more caused survival?

No. Survivors paid a higher average fare, but this result does not prove that paying more caused survival. Fare is related to other variables, particularly passenger class, which may also be associated with access, location and survival outcomes.

### 4. What is the limitation of the average age when approximately 20% of the values are missing?

The average is calculated only using passengers whose age is available. If missing ages are systematically associated with specific passenger groups, the observed mean may not accurately represent the complete population.


## Merge Operations

This section enriches the Titanic dataset by combining it with a lookup table containing the full names of the embarkation ports.

In [8]:
# Creation of port lookup table

port_lookup = pd.DataFrame({
    "Embarked" : ["C", "Q", "S"],
    "Port Name" : ["Cherbourg", "Queenstown", "Southampton"]
})

display(port_lookup)

,Embarked,Port Name
0,C,Cherbourg
1,Q,Queenstown
2,S,Southampton


In [9]:
# Verify: port-lookup shape, absence of NaN in Embarked column, Unicity of Embarked key, how many times each code is in the titanic dataset
print("The shape of the dataset created is: ", port_lookup.shape)
print("Number of missing values in Embarked column: ", port_lookup["Embarked"].isna().sum())
print("Unicity of Embarked key: ", port_lookup["Embarked"].is_unique)
display(titanic["Embarked"].value_counts(dropna=False))

The shape of the dataset created is:  (3, 2)
Number of missing values in Embarked column:  0
Unicity of Embarked key:  True


Embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64

### Why the uniqueness is important to be verified in the right dataframe?

In a many-to-one relationship, multiple rows from the left DataFrame may correspond to a single row in the right DataFrame. Therefore, the merge key must be unique in the right lookup table.

If the right DataFrame contains duplicated keys, a single left row may match multiple right rows, unexpectedly increasing the number of rows in the merged dataset.

In [10]:
# Execute a left merge between titanic and port_lookup datasets

titanic_enriched = titanic.merge(
    port_lookup,
    how="left",
    on="Embarked",
    validate="many_to_one",
    indicator=True
)

display(titanic_enriched.head())
print("Original shape: ", titanic.shape)
print("Merged shape: ", titanic_enriched.shape)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Port Name,_merge
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Southampton,both
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Cherbourg,both
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Southampton,both
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Southampton,both
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Southampton,both


Original shape:  (891, 12)
Merged shape:  (891, 14)


In [11]:
# Compute the distribution of the column '_merge' and find rows who did not find correspondence

merge_counts = (
    titanic_enriched["_merge"]
    .value_counts()
    .reindex(["both", "left_only", "right_only"], fill_value=0)
)

merge_distribution = pd.DataFrame({
    "row_count": merge_counts,
    "percentage": (
        merge_counts
        .div(len(titanic_enriched))
        .mul(100)
        .round(2)
    )
})

display(merge_distribution)

# Find the passenger without a correspondence

unmatched_passengers = titanic_enriched.loc[titanic_enriched["_merge"] == "left_only", 
                                            ["PassengerId", "Embarked", "Port Name", "_merge"]]
display(unmatched_passengers)


,row_count,percentage
_merge,,
both,889,99.78
left_only,2,0.22
right_only,0,0.00


,PassengerId,Embarked,Port Name,_merge
61,62,NaN,NaN,left_only
829,830,NaN,NaN,left_only


In [12]:
# Left join vs inner join

titanic_inner = titanic.merge(
    port_lookup,
    how="inner",
    on="Embarked",
    validate="many_to_one",
    indicator=True
)

print("Inner join dataset dimensions: ", titanic_inner.shape)

Inner join dataset dimensions:  (889, 14)


### Compare dimensions of inner join and left join

In the left join merge the number of rows was preserved while in the inner join, which only collects values that are common to left and right dataset, the two rows with NaN value have been eliminated.

An inner merge could be risky in this case because it doesn't count two rows only because they do not have an Embarked value and we possibly lose important values on other columns.

The left join preserves better the original dataset.

In [13]:
# From titanic_enriched, create a table containing, for every Port Name : passenger_count, survivor_count, survival_rate, average_fare

survival_enriched_summary = titanic_enriched.groupby("Port Name", dropna = False).agg(
    passenger_count = ("Port Name", "size"),
    survivor_count = ("Survived", "sum"),
    survival_rate = ("Survived", "mean"),
    average_fare = ("Fare", "mean")
)

survival_enriched_summary = survival_enriched_summary.round({
    "survival_rate" : 2,
    "average_fare" : 2
})

display(survival_enriched_summary)

,passenger_count,survivor_count,survival_rate,average_fare
Port Name,,,,
Cherbourg,168,93,0.55,59.95
Queenstown,77,30,0.39,13.28
Southampton,644,217,0.34,27.08
NaN,2,2,1.00,80.00


In [14]:
# Let's create a non valid table and try to repeat the merge

port_lookup_with_duplicate = pd.concat(
    [port_lookup, port_lookup.iloc[[0]]],
    ignore_index=True
)

display(port_lookup_with_duplicate)

,Embarked,Port Name
0,C,Cherbourg
1,Q,Queenstown
2,S,Southampton
3,C,Cherbourg


In [15]:
try:
    titanic.merge(
        port_lookup_with_duplicate,
        how="left",
        on="Embarked",
        validate="many_to_one",
        indicator=True
    )

except pd.errors.MergeError as error:
    print("Expected MergeError:")
    print(error)

Expected MergeError:
Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 Embarked
       C ...


### Which expectations was violated?

The expectation of unique keys in the right dataset.

### In which dataset is the duplicated key?

In the right dataset.

### Why trying to merge without validate would be risky?

Without `validate`, pandas does not check whether the merge keys respect the expected relationship. If the lookup table contains duplicated keys, the merge may still be completed and silently produce duplicated passenger rows.

### How would the number of rows change

The number of rows would grow, especially it would duplicate the number of rows of the duplicated key, if the key duplicated is Cherbourg, every passenger embarked from Cherbourg will be duplicated in the new dataset.

## Observations

### 1. Why is a left merge more suitable than an inner merge in this case?

It is more suitable because, given the presence of null values in the `Embarked` column an inner join would not consider these two rows in the final dataset.

### 2. Why should the number of rows remain unchanged?

The merge follows a many-to-one relationship: each Titanic passenger can match at most one row in the port lookup table. Since a left merge also preserves unmatched rows from the left DataFrame, every original passenger should appear exactly once in the merged dataset.

### 3. What information does the `_merge` column provide?

The `_merge` column indicates whether the merge key for each resulting row was found in both DataFrames, only in the left DataFrame or only in the right DataFrame. It is therefore useful for identifying unmatched keys and validating the merge.

### 4. Why is `validate = "many_to_one` useful?

It verifies that each merge key appears at most once in the right DataFrame. This prevents duplicated lookup keys from silently multiplying passenger rows in the merged dataset.

### 5. What could cause a merge to unexpectedly increase the number of rows?

The presence of duplicated keys in the right dataset (in this case), if the `validate = "many_to_one` is not specified, could count twice the times the duplicated keys in the right dataset and increase in this way the number of rows.